In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV

# Load dataset (prepared)
df = pd.read_csv("../data/cleaned/final_prepared_dataset.csv")
df.head()

,name,GW,total_points,minutes,goals_scored,assists,bonus,bps,clean_sheets,creativity,...,team_x_Newcastle,team_x_Norwich,team_x_Nott'm Forest,team_x_Sheffield Utd,team_x_Southampton,team_x_Spurs,team_x_Watford,team_x_West Brom,team_x_West Ham,team_x_Wolves
0,Aaron Connolly,1,1,45,0,0,0,-3,0,0.3,...,False,False,False,False,False,False,False,False,False,False
1,Aaron Connolly,2,8,89,1,0,2,27,1,11.3,...,False,False,False,False,False,False,False,False,False,False
2,Aaron Connolly,3,2,73,0,0,0,2,0,12.1,...,False,False,False,False,False,False,False,False,False,False
3,Aaron Connolly,4,2,65,0,0,0,7,0,0.3,...,False,False,False,False,False,False,False,False,False,False
4,Aaron Connolly,5,4,12,0,1,0,13,0,10.3,...,False,False,False,False,False,False,False,False,False,False


In [7]:
X = df.drop(columns=["name", "GW", "upcoming_total_points"], errors='ignore')
y = df["upcoming_total_points"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [8]:
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)   
rmse = np.sqrt(mse)
print(f"Random Forest Regressor Performance:")
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R²: {r2}")  

Random Forest Regressor Performance:
MAE: 1.3468250702876932
MSE: 5.243294975569808
RMSE: 2.289824223727622
R²: 0.3069948839644464


In [9]:
from scipy.stats import randint, uniform
param_dist = {
    "n_estimators": randint(100, 1001),
    "max_depth": [None, 5, 10, 20, 30],
    "max_features": ["sqrt", "log2", None, 0.2, 0.5],
    "min_samples_split": randint(2, 21),
    "min_samples_leaf": randint(1, 21),
    "bootstrap": [True, False],
    "min_impurity_decrease": uniform(0.0, 0.001),
    "ccp_alpha": uniform(0.0, 0.01)
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)
search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,

    random_state=42,
    verbose=1
)
search.fit(X_train, y_train)
print(search.best_params_)


Fitting 3 folds for each of 10 candidates, totalling 30 fits
{'bootstrap': False, 'ccp_alpha': np.float64(0.0033370861113902186), 'max_depth': 10, 'max_features': 0.5, 'min_impurity_decrease': np.float64(0.0009699098521619943), 'min_samples_leaf': 12, 'min_samples_split': 7, 'n_estimators': 485}


In [10]:
params = {
    "bootstrap": False,
    "ccp_alpha": np.float64(0.0033370861113902186),
    "max_depth": 10,
    "max_features": 0.5,
    "min_impurity_decrease": np.float64(0.0009699098521619943),
    "min_samples_leaf": 12,
    "min_samples_split": 7,
    "n_estimators": 485
}

rf_tuned = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
rf_tuned.fit(X_train, y_train)
y_pred_tuned = rf_tuned.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_tuned)
mse = mean_squared_error(y_test, y_pred_tuned)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_tuned)


print(f"MAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")

MAE: 1.3495, MSE: 5.2309, RMSE: 2.2871, R²: 0.3086
